# 03: Supervisor e o grafo do pipeline

Aqui os agentes dos notebooks 01 e 02 viram um **pipeline**: um `StateGraph` do LangGraph em que o
**Supervisor** decide quem trabalha a cada passo e todos os agentes voltam para ele.

No estudo inicial do padrão, o supervisor era um agente que escolhia tools sozinho. Aqui ele é uma **função Python**:
regras objetivas rodam em código (os *guards*) e só o julgamento que precisa de contexto vai para o LLM.

In [1]:
import sys
import warnings
from pathlib import Path

# Sobe pastas até achar a raiz do projeto (onde está o pyproject.toml), de onde quer que o notebook seja aberto
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(RAIZ))

from dotenv import load_dotenv
_ = load_dotenv(RAIZ / ".env")

# Aviso de depreciação interno do LangGraph, sem relação com o projeto: só polui a saída.
# O filtro vem depois do import do langchain_core, que instala o filtro dele ao ser importado.
from langchain_core._api.deprecation import LangChainPendingDeprecationWarning
warnings.filterwarnings("ignore", category=LangChainPendingDeprecationWarning)
warnings.filterwarnings("ignore", message="Function wrap_gemini is in beta")

## 1. O desenho do grafo

Nós: `Supervisor`, os três workers (`YFinance`, `Analyzer`, `Investigator`) e o `Consolidador`. As setas
pontilhadas saem do Supervisor (ele escolhe o próximo); as cheias voltam a ele. Do Consolidador, o fluxo vai ao fim.

In [2]:
from fluxo_financeiro import app

print(app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	Supervisor(Supervisor)
	YFinance(YFinance)
	Analyzer(Analyzer)
	Investigator(Investigator)
	Consolidador(Consolidador)
	__end__([<p>__end__</p>]):::last
	Analyzer --> Supervisor;
	Investigator --> Supervisor;
	Supervisor -.-> Analyzer;
	Supervisor -.-> Consolidador;
	Supervisor -.-> Investigator;
	Supervisor -.-> YFinance;
	YFinance --> Supervisor;
	__start__ --> Supervisor;
	Consolidador --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 2. O estado que circula

O grafo carrega um dicionário (`MessagesState`) que cada nó lê e atualiza:

| Campo | O que guarda |
|---|---|
| `messages` | o histórico; cada mensagem de agente tem `name=` dizendo quem falou |
| `next_agent` | a decisão do Supervisor (ou `PARALELO` via guard) |
| `loop_count` | quantas vezes o Supervisor passou (soma 1 por passagem) |
| `decidido_por` | quem decidiu: `guard`, `llm` ou `hard_stop` |
| `supervisor_reasoning` | a justificativa da decisão |
| `sinais_anomalia` | sinais calculados em código pelo nó do YFinance (notebook 10) |
| `relatorio` | o relatório estruturado do Consolidador (notebook 11) |
| `dados_mercado` | texto coletado pelo YFinance |
| `noticias` | seção de notícias da empresa extraída da resposta do Analyzer |
| `contexto_setor` | seção de setor extraída da resposta do Analyzer |
| `diagnostico` | diagnóstico causal do Investigator (se executado) |

## 3. O pipeline de etapas e o paralelismo

| Etapa | Situação | Destino | Quem decide |
|---|---|---|---|
| 1 | Ainda não há dados | YFinance | LLM |
| 2 (sem sinais) | YFinance coletou, Analyzer não rodou | Analyzer (notícias e setor numa passagem só) | guard |
| 2 (com sinais) | YFinance coletou e há sinais de anomalia | Analyzer e Investigator em paralelo (`PARALELO`) | guard |
| 3 | O setor vem na mesma resposta do Analyzer | (não há passagem separada) | código |
| 4 | Analyzer concluiu e não há sinais de anomalia | Investigator ou FINISH | LLM |
| fim | Investigator concluiu (ou desistiu) e Analyzer resolvido | FINISH | guard |
| fim | `loop_count` chegou a 6 | FINISH | hard stop |

Todo FINISH passa pelo **Consolidador** antes de acabar: quem manda para lá é o `router`, não o LLM.
A antiga Etapa 3 deixou de existir: o setor vem na seção `## Contexto do setor` da resposta única do Analyzer.
Com sinais de anomalia, o Investigator roda junto com o Analyzer na Etapa 2 em paralelo.
Na etapa 4 sem sinais, o LLM responde à pergunta do prompt: *"Um investidor ficaria com dúvida de POR QUE
esse número aconteceu?"*

In [3]:
from fluxo_financeiro.graph.supervisor import SUPERVISOR_PROMPT

inicio = SUPERVISOR_PROMPT.index("<pipeline>")
fim = SUPERVISOR_PROMPT.index("</pipeline>") + len("</pipeline>")
print(SUPERVISOR_PROMPT[inicio:fim])

<pipeline>
Siga esta ordem rigorosamente:
Etapa 1 (dados financeiros): se não há mensagem do YFinance, rotear para 'YFinance'.
Etapa 2 (notícias e setor): se há YFinance mas nenhuma resposta do Analyzer, rotear para 'Analyzer' (já tratado automaticamente pelo guard). O Analyzer pesquisa as notícias da empresa e o contexto do setor numa passagem só. Se a mensagem do YFinance trouxer "SINAIS DE ANOMALIA", o guard manda o Investigator junto, ao mesmo tempo que o Analyzer.
Etapa 3 (contexto do setor): não é mais uma passagem separada. O setor vem na mesma resposta do Analyzer, na seção '## Contexto do setor'. Não mande o Analyzer de novo por causa do setor, mesmo que essa seção falte.
Etapa 4 (investigação condicional): depois da resposta do Analyzer:
  Se a mensagem do YFinance trouxer "SINAIS DE ANOMALIA", o Investigator já rodou junto com o Analyzer (guard da Etapa 2).
  Você só decide aqui quando não há sinais calculados; mesmo assim pode haver algo que mereça investigação.
  Leia o co

## 4. Uma análise completa, passo a passo (WEGE3)

`app.stream` devolve a atualização de cada nó assim que ele termina. Imprimimos, para cada passo, quem
rodou, o que o Supervisor decidiu e quem decidiu. Esta célula faz a análise inteira de verdade
(cerca de 20 a 30 segundos e alguns centavos de dólar).

In [4]:
from langchain_core.messages import HumanMessage

passos = []
final = None
for evento in app.stream({"messages": [HumanMessage("Analise a ação WEGE3")]}):
    for no, update in evento.items():
        update = update or {}
        if no == "Supervisor":
            print(f"[Supervisor] -> {update.get('next_agent')}  (decidido por: {update.get('decidido_por')})")
            if update.get("supervisor_reasoning"):
                print(f"   motivo: {update['supervisor_reasoning'][:160]}")
            passos.append(f"Supervisor->{update.get('next_agent')}")
        else:
            texto = update["messages"][-1].text if update.get("messages") else ""
            print(f"[{no}] {len(texto)} caracteres: {texto[:160]!r}")
            passos.append(no)
            if no == "Consolidador":
                final = texto

print("\nCaminho:", " | ".join(passos))

2026-11-08 21:57:49,249 - fluxo_financeiro - INFO - [supervisor] inicio — loop_count=0


2026-11-08 21:57:50,885 - fluxo_financeiro - INFO - [supervisor] decisao=YFinance | tarefa='Colete os dados financeiros, histórico de preços e métricas fundamentais para a ação WEGE3.'


2026-11-08 21:57:50,887 - fluxo_financeiro - DEBUG - [router] YFinance → YFinance


2026-11-08 21:57:50,889 - fluxo_financeiro - INFO - [node] YFinance: invocando agente (2 msgs)


[Supervisor] -> YFinance  (decidido por: llm)
   motivo: O usuário solicitou a análise da ação WEGE3 e ainda não há dados financeiros coletados. O próximo passo obrigatório é acionar o YFinance para buscar esses dados


2026-11-08 21:57:51,971 - fluxo_financeiro - INFO - [tool] buscar_dados_acao_b3: ticker=WEGE3.SA


... (28 outputs left out)

2026-11-08 21:58:11,145 - fluxo_financeiro - INFO - [node] Consolidador: consolidando (5 msgs)


[Supervisor] -> FINISH  (decidido por: llm)
   motivo: Os dados financeiros e as notícias do setor/empresa já foram coletados. O Piotroski F-Score de 5/9 e os indicadores apontam solidez sem anomalias críticas inexp


2026-11-08 21:58:13,390 - fluxo_financeiro - INFO - [node] Consolidador: relatório pronto (1807 chars)


[Consolidador] 1807 caracteres: '# Relatório: WEGE3\n\n## Resumo\nA WEG (WEGE3) apresenta alta rentabilidade com ROE de 29,9% e forte geração de caixa operacional, negociando a um P/L de 32,3x (qu'

Caminho: Supervisor->YFinance | YFinance | Supervisor->Analyzer | Analyzer | Supervisor->FINISH | Consolidador


## 5. A resposta final

A última mensagem do histórico é o relatório em markdown do Consolidador.

In [5]:
from IPython.display import Markdown

Markdown(final)

# Relatório: WEGE3

## Resumo
A WEG (WEGE3) apresenta alta rentabilidade com ROE de 29,9% e forte geração de caixa operacional, negociando a um P/L de 32,3x (quantos anos de lucro pagam o preço da ação). A empresa destaca-se como líder isolada em valor de mercado no setor de Máquinas e Equipamentos, mas enfrenta pressões pontuais de custos nas margens e aumento da dívida corrente.

## Indicadores
- Preço Atual: R$ 48.19
- Faixa de 52 semanas: R$ 41.15 a R$ 54.53
- Valor de mercado: R$ 210.0 bi
- LPA: R$ 1.49
- P/L: 32.3x
- ROE: 29.9%
- DY: 3.22%
- Sharpe Ratio: 0.18
- Sortino Ratio: 0.28
- Max Drawdown: -22.5%
- VaR 95% (diário): -3.14%
- Beta (vs IBOV): 0.29
- Alpha (anual): 6.6%
- Piotroski F-Score: 5/9
- Net Income: R$ 6.38B
- Operating Cash Flow: R$ 6.45B
- Total Assets: +2.8%
- Divida corrente: +12.5%
- Current Ratio: 1.55
- Gross Margin: 33.5%
- SELIC (meta): 13.75% a.a.
- CDI: 13.65% a.a.
- IPCA acumulado em 12 meses: 4.58%

## Notícias e setor
Inserida no subsetor de Máquinas e Equipamentos dentro do segmento de Bens Industriais da B3, a WEG destaca-se como a líder isolada do setor em valor de mercado, com R$ 210 bilhões. A empresa anunciou a aquisição de um terreno na Índia por aproximadamente R$ 44 milhões para expandir sua pegada internacional. O setor tem sido impactado por pressões inflacionárias, oscilações nos preços de commodities metálicas, custos logísticos e restrições decorrentes de tarifas comerciais internacionais impostas pelos Estados Unidos.

## Riscos
- Divida corrente aumentou (+12.5%)
- Current Ratio piorou (1.55 vs 1.76)
- Gross Margin caiu (33.5% vs 33.7%)
- Pressões pontuais de custos nas margens, impulsionadas pela alta de matérias-primas como cobre e aço
- Reflexos operacionais de novas tarifas comerciais nos Estados Unidos e custos de ramp-up


## O que observar

- No caminho da WEGE3, a etapa 4 normalmente termina em FINISH sem Investigator: sem sinais de anomalia, o LLM não viu um porquê a investigar. A VALE3 tem sinais, e lá o Investigator entra em paralelo com o Analyzer por guard na Etapa 2 (notebooks 10, 11 e 19).
- O passo 2 diz `decidido por: guard`: envio para o Analyzer numa passagem só, sem chamada ao LLM. O primeiro passo (para YFinance) e o terceiro (para FINISH) dizem `llm`.
- O Consolidador não aparece como decisão do Supervisor: o `router` manda todo FINISH para ele.
- São apenas 3 passagens do Supervisor numa análise sem Investigator (`loop_count = 3`, antes eram 4); com Investigator em paralelo na VALE3 também são 3 passagens (`loop_count = 3`, antes eram 5). O hard stop em 6 é margem de segurança.